In [36]:
!pip install scikit-surprise

Defaulting to user installation because normal site-packages is not writeable
  Using cached scikit_surprise-1.1.4.tar.gz (154 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [44 lines of output]
      Compiling surprise/similarities.pyx because it changed.
      Compiling surprise/prediction_algorithms/matrix_factorization.pyx because it changed.
      Compiling surprise/prediction_algorithms/optimize_baselines.pyx because it changed.
      Compiling surprise/prediction_algorithms/slope_one.pyx because it changed.
      Compiling surprise/prediction_algorithms/co_clustering.pyx because it changed.
      [1/5] Cythonizing surprise/prediction_algorithms/co_clustering.pyx
      
      Error compiling Cython file:
      ------------------------------------------------------------
      ...
              self.avg_cltr_i = avg_cltr_i
              self.avg_cocltr = avg_cocltr
      
              return self
      
          def compute_averages(self, np.ndarray[np.int_t] cltr_u,
                                                   ^
  

In [37]:
import pandas as pd

df = pd.read_csv("coursera.csv")
df.head()

,partner,course,skills,rating,reviewcount,level,certificatetype,duration,crediteligibility
0,Google,Google Cybersecurity,"{"" Network Security"","" Python Programming"","" L...",4.8,16.4k,Beginner,Professional Certificate,3 - 6 Months,False
1,Google,Google Data Analytics,"{"" Data Analysis"","" R Programming"","" SQL"","" Bu...",4.8,133.4k,Beginner,Professional Certificate,3 - 6 Months,True
2,Google,Google Project Management:,"{"" Project Management"","" Strategy and Operatio...",4.8,97.3k,Beginner,Professional Certificate,3 - 6 Months,True
3,Google,Google Digital Marketing & E-commerce,"{"" Digital Marketing"","" Marketing"","" Marketing...",4.8,21.4k,Beginner,Professional Certificate,3 - 6 Months,False
4,Google,Google IT Support,"{"" Computer Networking"","" Network Architecture...",4.8,181.4k,Beginner,Professional Certificate,3 - 6 Months,True


In [38]:
# Remove unnecessary column
df = df.drop(columns=['Unnamed: 0'], errors='ignore')

# Check for missing values
print(df.isnull().sum())

# Convert rating column to float
df['rating'] = df['rating'].astype(float)

df.head()


partner                0
course                 0
skills                51
rating               146
reviewcount          146
level                135
certificatetype       23
duration              23
crediteligibility      0
dtype: int64


,partner,course,skills,rating,reviewcount,level,certificatetype,duration,crediteligibility
0,Google,Google Cybersecurity,"{"" Network Security"","" Python Programming"","" L...",4.8,16.4k,Beginner,Professional Certificate,3 - 6 Months,False
1,Google,Google Data Analytics,"{"" Data Analysis"","" R Programming"","" SQL"","" Bu...",4.8,133.4k,Beginner,Professional Certificate,3 - 6 Months,True
2,Google,Google Project Management:,"{"" Project Management"","" Strategy and Operatio...",4.8,97.3k,Beginner,Professional Certificate,3 - 6 Months,True
3,Google,Google Digital Marketing & E-commerce,"{"" Digital Marketing"","" Marketing"","" Marketing...",4.8,21.4k,Beginner,Professional Certificate,3 - 6 Months,False
4,Google,Google IT Support,"{"" Computer Networking"","" Network Architecture...",4.8,181.4k,Beginner,Professional Certificate,3 - 6 Months,True


In [39]:
df['features'] = df['course'] + " " + \
                 df['partner'] + " " + \
                 df['level'] + " " + \
                 df['certificatetype']

df[['course','features']].head()

,course,features
0,Google Cybersecurity,Google Cybersecurity Google Beginner Profess...
1,Google Data Analytics,Google Data Analytics Google Beginner Profes...
2,Google Project Management:,Google Project Management: Google Beginner P...
3,Google Digital Marketing & E-commerce,Google Digital Marketing & E-commerce Google B...
4,Google IT Support,Google IT Support Google Beginner Profession...


In [40]:
df['features'] = df['course'] + " " + df['skills'] + " " + df['level']
df['features'] = df['features'].fillna('')

In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english')

feature_matrix = vectorizer.fit_transform(df['features'])

feature_matrix.shape

(1139, 1377)

In [42]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(feature_matrix)

similarity_matrix

array([[1.        , 0.17936058, 0.19327817, ..., 0.        , 0.        ,
        0.        ],
       [0.17936058, 1.        , 0.2053682 , ..., 0.        , 0.        ,
        0.        ],
       [0.19327817, 0.2053682 , 1.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]], shape=(1139, 1139))

In [56]:
def recommend_courses(course_name):

    idx = df[df['course'] == course_name].index[0]

    similarity_scores = list(enumerate(similarity_matrix[idx]))

    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    top_courses = similarity_scores[1:6]

    for i in top_courses:
        print(df.iloc[i[0]]['course'])

In [59]:
df['course']

0                                    Google Cybersecurity
1                                   Google Data Analytics
2                              Google Project Management:
3                   Google Digital Marketing & E-commerce
4                                       Google IT Support
                              ...                        
1134    Post Graduate Certificate in Cloud Computing A...
1135           Business Essentials University Certificate
1136    Post Graduate Certificate in Strategic Supply ...
1137    Power Electronics and Motors for Electric Vehi...
1138    Entrepreneurship and Strategic Innovation Grad...
Name: course, Length: 1139, dtype: str

In [57]:
recommend_courses("Machine Learning")

Machine Learning and Reinforcement Learning in Finance
Advanced Learning Algorithms
Unsupervised Learning, Recommenders, Reinforcement Learning
Machine Learning for All
Introduction to TensorFlow for Artificial Intelligence, Machine Learning, and Deep Learning


In [60]:
course_to_recommend = "Google Data Analytics"
if course_to_recommend in df['course'].values:
    recommend_courses(course_to_recommend)
else:
    print(f"Course '{course_to_recommend}' not found in the dataset. Please provide an existing course title.")

Google データアナリティクス
Google Data Analytics (PT)
Foundations: Data, Data, Everywhere
Data Analysis and Presentation Skills: the PwC Approach
Excel to MySQL: Analytic Techniques for Business


In [61]:
recommend_courses("Python for Everybody")

Meta Database Engineer
SQL for Data Science
Databases and SQL for Data Science with Python
Data Engineering Foundations
Learn SQL Basics for Data Science


In [62]:
recommend_courses("Business Foundations")

Introduction to Finance and Accounting
Foundations of Management
Fundamentals of Accounting
Business Value and Project Management
Financial Analysis - Skills for Success


In [70]:
def recommend_courses(course_name):
    course_name = course_name.lower()

    df['course'] = df['course'].str.lower()

    matches = df[df['course'].str.contains(course_name)]

    if matches.empty:
        print("❌ Course not found in dataset")
        print("Try one of these:")
        print(df['course'].head(10).values)
        return

    idx = matches.index[0]

    print("✅ Using:", df.iloc[idx]['course'])

    similarity_scores = list(enumerate(similarity_matrix[idx]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    print("\nRecommended Courses:\n")

    for i in similarity_scores[1:6]:
        print(df.iloc[i[0]]['course'])

In [71]:
recommend_courses("Google Data Analytics")

✅ Using: google data analytics

Recommended Courses:

google データアナリティクス
google data analytics (pt)
foundations: data, data, everywhere
data analysis and presentation skills: the pwc approach
excel to mysql: analytic techniques for business


In [73]:
recommend_courses("Deep Learning")

✅ Using: deep learning

Recommended Courses:

introduction to machine learning
deeplearning.ai tensorflow developer
neural networks and deep learning
machine learning
introduction to tensorflow for artificial intelligence, machine learning, and deep learning


In [78]:
def recommend_courses(course_name, num_recommendations=5):

    course_name = course_name.lower()

    df['course'] = df['course'].fillna('').str.lower()

    matches = df[df['course'].str.contains(course_name)]

    if matches.empty:
        print("❌ Course not found in dataset")
        return

    idx = matches.index[0]

    similarity_scores = list(enumerate(similarity_matrix[idx]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    top_courses = similarity_scores[1:num_recommendations+1]

    recommended_indices = [i[0] for i in top_courses]

    result = df.iloc[recommended_indices][[
        'course',
        'partner',
        'rating',
        'level'
    ]]

    result = result.reset_index(drop=True)
    result.index = result.index + 1

    return result

In [79]:
recommend_courses("Machine Learning")

,course,partner,rating,level
1,machine learning and reinforcement learning in...,New York University,3.7,Intermediate
2,advanced learning algorithms,DeepLearning.AI,4.9,Beginner
3,"unsupervised learning, recommenders, reinforce...",DeepLearning.AI,4.9,Beginner
4,machine learning for all,University of London,4.7,Beginner
5,introduction to tensorflow for artificial inte...,DeepLearning.AI,4.8,Intermediate


In [80]:
recommend_courses("operating system")

,course,partner,rating,level
1,introduction to hardware and operating systems,IBM,4.8,Beginner
2,introduction to computers and operating system...,Microsoft,4.8,Beginner
3,"cybersecurity roles, processes & operating sys...",IBM,4.6,Beginner
4,advanced embedded linux development,University of Colorado Boulder,4.0,Intermediate
5,introduction to systems engineering,UNSW Sydney (The University of New South Wales),4.7,Mixed


In [22]:
!git clone https://github.com/Jolsina/course_guide

Cloning into 'course_guide'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 18 (delta 4), reused 16 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 62.49 KiB | 1.89 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [25]:
!mv course_recommender.ipynb course_guide/

mv: cannot stat 'course_recommender.ipynb': No such file or directory
